In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

crime_data = pd.read_csv('data/crimedata.csv')
sociodata = pd.read_csv('data/sociodata.csv')

In [ ]:
import requests
import plotly.express as px
import pandas as pd

shapefile = requests.get(
    "https://data.cityofchicago.org/resource/igwz-8jzy.json"
).json()

geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": row["the_geom"],
            "properties": {k: v for k, v in row.items() if k != "the_geom"},
        }
        for row in shapefile
        if row.get("the_geom") and row["the_geom"].get("coordinates")
    ],
}

In [ ]:
mapping = {sociodata['Community Area Number'][i]: sociodata['COMMUNITY AREA NAME'][i] for i in range(len(sociodata))}
crime_data['Community Area'] = crime_data['Community Area'].map(mapping)

In [ ]:
P_crime = crime_data['Primary Type'].value_counts(normalize=True)
P_crime_given_district = crime_data.groupby('Community Area')['Primary Type'].value_counts(normalize=True)
ratio = P_crime_given_district.div(P_crime, level='Primary Type')
ratio = ratio.reset_index(name='ratio')

In [ ]:
sd = pd.read_csv('./data/sociodata.csv')
print(f"{len(sd)} rows, {len(sd.columns)} columns")
# sd.describe()
sd.sample(frac=1).reset_index(drop=True).head(5)

In [ ]:
def _get_shapes_file() -> dict:
    """Return geojson for chicago community areas"""
    # import shapefile for community areas
    shapefile = requests.get("https://data.cityofchicago.org/resource/igwz-8jzy.json").json()

    # And convert them to proper geojson format
    geojson = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "geometry": row["the_geom"],
                "properties": {k: v for k, v in row.items() if k != "the_geom"},
            }
            for row in shapefile
            if row.get("the_geom") and row["the_geom"].get("coordinates")
        ],
    }
    return geojson

def _align_naming_schemes(df: pd.DataFrame) -> pd.DataFrame:
    """Align the naming schemes of the comminity areas (city of chicago misspelled some names in the census data... smh)"""
    df = df.copy()
    # Spell correction
    name_fixes = {
        "MONTCLAIRE": "MONTCLARE",
        "WASHINGTON HEIGHT": "WASHINGTON HEIGHTS",
        "O'HARE": "OHARE",
    }
    df["COMMUNITY AREA NAME"] = df["COMMUNITY AREA NAME"].str.upper().replace(name_fixes)
    # Exclude the "total" row
    df = df[df["COMMUNITY AREA NAME"] != "CHICAGO"]
    return df

def plot_income_choropleth(col: str, filename: str, plot_data: pd.DataFrame, geojson: dict) -> None:
    import plotly.express as px
    import plotly.graph_objects as go

    fig = go.Figure(
        data=px.choropleth_map(
            plot_data,
            geojson=geojson,
            locations="COMMUNITY AREA NAME",
            color=col,
            featureidkey="properties.community",
            color_continuous_scale="Magma",
            map_style="carto-positron",
            zoom=9,
            center={"lat": 41.8781, "lon": -87.6298},
            opacity=0.6,
        )
    )
    fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
    fig.write_html(f"./../docs/figures/{filename}", include_plotlyjs="cdn")
    fig.show()

geojson = _get_shapes_file()
plot_data = _align_naming_schemes(sd)

# # Generate choropleth plots for the 4 main columns
# plot_income_choropleth("HARDSHIP INDEX", "hardship_choropleth.html", plot_data, geojson)
# plot_income_choropleth("PER CAPITA INCOME ", "income_choropleth.html", plot_data, geojson) # Yes, trailing space is on purpose... smh
# plot_income_choropleth("PERCENT HOUSEHOLDS BELOW POVERTY", "poverty_choropleth.html", plot_data, geojson)
# plot_income_choropleth("PERCENT AGED 16+ UNEMPLOYED", "unemployment_choropleth.html", plot_data, geojson)

In [ ]:
crime_by_type = (
    crime_data
    .groupby(["Community Area", "Primary Type"])
    .size()
    .reset_index(name="count")
)

crime_pivot = crime_by_type.pivot(
    index="Community Area",
    columns="Primary Type",
    values="count"
).fillna(0)

crime_pivot = crime_pivot.reset_index()

merged_types = sociodata.merge(
    crime_pivot,
    left_on="COMMUNITY AREA NAME",
    right_on="Community Area",
    how="inner"
)

# 1. Ensure only crime columns are used
crime_columns = crime_pivot.columns.tolist()
crime_columns.remove("Community Area")

# 2. Compute correlation manually (avoids .corr() issues)
corr_with_hardship = merged_types[crime_columns].apply(
    lambda x: x.corr(merged_types["HARDSHIP INDEX"])
)

# 3. Drop any NaNs (some crime types may be constant = no variance)
corr_with_hardship = corr_with_hardship.dropna()

# 4. Convert to clean DataFrame for Plotly
plot_df = corr_with_hardship.sort_values().reset_index()
plot_df.columns = ["Crime Type", "Correlation"]

# 5. Plot (fully aligned)
fig = px.bar(
    plot_df,
    x="Correlation",
    y="Crime Type",
    orientation="h",
    title="Correlation of Crime Types with Hardship Index",
)

fig.update_layout(
    height=800,  # improves readability
    yaxis_title="Crime Type",
    xaxis_title="Correlation"
)

filename = "crime_hardship_correlation.html"

fig.write_html(f"./../docs/figures/{filename}", include_plotlyjs="cdn")

fig.show()

This plot shows the correlation between each crime type and the hardship index across Chicago community areas. Each bar represents how strongly a specific crime type is associated with neighborhoods that have higher or lower hardship levels. Positive values indicate that a crime type tends to occur more frequently in areas with higher hardship, while negative values suggest it is more common in lower hardship areas. Values close to zero indicate little to no relationship between the crime type and hardship index.

It’s important to remember that this plot shows correlation, not causation. That means it only describes how crime types and hardship levels vary together across areas—it does not prove that hardship causes certain crimes, or that crime causes hardship. Other factors like population size, reporting practices, and neighborhood characteristics can influence these relationships, so the results should be interpreted as patterns of association rather than direct cause-and-effect relationships.

In [ ]:
# import statsmodels.api as sm

# r2_values = {}

# for crime in crime_columns:
#     X = sm.add_constant(merged_types["HARDSHIP INDEX"])
#     y = merged_types[crime]

#     model = sm.OLS(y, X, missing="drop").fit()
#     r2_values[crime] = model.rsquared

# import pandas as pd

# r2_df = pd.DataFrame.from_dict(
#     r2_values,
#     orient="index",
#     columns=["R_squared"]
# ).sort_values("R_squared")

# import plotly.express as px

# fig = px.bar(
#     r2_df,
#     x="R_squared",
#     y=r2_df.index,
#     orientation="h",
#     title="R²: How Much Hardship Explains Each Crime Type",
# )

# # fig.update_layout(height=800)
# fig.show()